# Encode Categorical Features

### Learning Objectives

We learn to:

1. Identify when to use categorical feature encoding
2. Distinguish between `OrdinalEncoder` and `OneHotEncoder`
3. Show why one-hot encoding could be a problem; and explain how to handle that problem

Categorical data must be encoded numerically for use in most machine learning algorithms.

**Ordinal** categories (e.g. sizes: small, medium, large, or ranks: 1st, 2nd, 3rd) have a natural order and can be encoded as integers (1, 2, 3) using `OrdinalEncoder`.

**Nominal** categories have no inherent order. They are often encoded as binary columns (0/1) for each distinct value using `OneHotEncoder`.


![](../assets/ohe_vs_oe.png)

In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

## OrdinalEncoder

In [2]:
from sklearn.preprocessing import OrdinalEncoder

# Specify the order of categories
categories = [
    ['low', 'medium', 'high'], # categories of first feature
    ['1st', '2nd', '3rd'],     # categories of second feature
]

encoder = OrdinalEncoder(
    categories=categories,
    handle_unknown='use_encoded_value', # means that if we encounter an unknown category, we will encode it as a specific value
    unknown_value=-1
)
# We want a pandas DataFrame as output rather than a NumPy array (default)
encoder = encoder.set_output(transform='pandas')

In [3]:
df = pd.DataFrame({
    'risk': ['low', 'medium', 'low', 'low', 'high'],
    'class': ['1st', '3rd', '2nd', '1st', '3rd'],
})
df

,risk,class
0,low,1st
1,medium,3rd
2,low,2nd
3,low,1st
4,high,3rd


In [4]:
encoder.fit_transform(df)

,risk,class
0,0.0,0.0
1,1.0,2.0
2,0.0,1.0
3,0.0,0.0
4,2.0,2.0


## OneHotEncoder

In [5]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder instance
encoder = OneHotEncoder(
    handle_unknown='infrequent_if_exist',
    sparse_output=False,    # <-- output is a dense array
)
encoder.set_output(transform='pandas')

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'infrequent_if_exist'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",No

In [6]:
df_train = pd.DataFrame({
    'color': ['Red', 'Blue', 'Green', 'Green']
})
df_train

,color
0,Red
1,Blue
2,Green
3,Green


In [7]:
# Fit and transform the data
encoder.fit_transform(df_train)

,color_Blue,color_Green,color_Red
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0


In [8]:
df_test = pd.DataFrame({
    'color': ['Blue', 'Green', 'dragonfruit']
})
df_test

,color
0,Blue
1,Green
2,dragonfruit


In [9]:
# 4. Transform
result = encoder.transform(df_test)
result

,color_Blue,color_Green,color_Red
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,0.0


### Memory cost of one-hot encoding

Each distinct category gets its own column, which can use a lot of memory. We can limit this by treating only categories that appear at least `min_frequency` times as separate columns, and by capping the total number of encoded columns with `max_categories`.


"For example, here are the extra columns and memory from one-hot encoding categorical variables in the **[Ames Housing dataset](https://www.openml.org/search?type=data&status=active&id=41211&sort=runs)**:"


| **Metric** | **Before encoding** | **After one-hot encoding** | **Increase** |
| ---: | ---: | ---: | ---: |
| **Number of columns** | ~46 | ~318 | **~6.9x** |
| **Size** | ~0.15 MB | ~7.10 MB | **~47.3x** |

For details on reproducing this, see: [Demonstrating the memory cost of one-hot encoding](ohe_problem_demo.ipynb).


### Mitigation

* Use **`min_frequency`** to reduce the number of columns by grouping infrequent categories into a single column.
* Use **`max_categories`** to cap the total number of output columns (including the infrequent-category column).

**OneHotEncoder** can group infrequent categories into one column per feature, as summarized below:

| **Parameter** | **Type** | **Rule** | **Description** |
| --- | --- | --- | --- |
| **`min_frequency`** | `int` | $\\ge 1$ | Categories that appear fewer than this many times are treated as infrequent. |
| _ | `float` | $(0.0, 1.0)$ | Categories that appear in less than this fraction of samples are treated as infrequent. |
| **`max_categories`** | `int` | $> 1$ | Maximum total number of output features per input feature, including the infrequent category. |
| _ | `None` | default | No limit on the number of output features. |


In [10]:
X = np.array([
    ['cat'] * 20 +
    ['rabbit'] * 10 +
    ['snake'] * 6 +
    ['dragon'] * 3 +
    ['dinosaur'] * 2
], dtype=str).T
X.shape

(41, 1)

Reminder: multiplying a list by an integer repeats its elements; adding two lists concatenates them into a new list.


In [11]:
enc = preprocessing.OneHotEncoder(
    min_frequency=6,
    max_categories=3,
    handle_unknown='infrequent_if_exist',
    sparse_output=False,
)
enc.set_output(transform='pandas')
enc.fit(X)

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'infrequent_if_exist'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",6


- Note how both `'dragon'` and `'dinosaur'` are mapped to the same encoding `[0., 0., 1.]` (the shared encoding for all infrequent categories).
- Because we set `max_categories=3`, `'snake'` does not get its own column even though it meets `min_frequency`; the third column is reserved for the combined infrequent categories.


In [12]:
enc.transform(np.array([
    ['rabbit'],
    ['rabbit'],
    ['cat'],
    ['snake'],
    ['dragon'],
    ['dinosaur'],
]))

,x0_cat,x0_rabbit,x0_infrequent_sklearn
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
5,0.0,0.0,1.0


In [13]:
print("Categories:", enc.categories_)
print("Infrequent categories:", enc.infrequent_categories_)

Categories: [array(['cat', 'dinosaur', 'dragon', 'rabbit', 'snake'], dtype='<U8')]
Infrequent categories: [array(['dinosaur', 'dragon', 'snake'], dtype='<U8')]


---

- **Geocoding** produces latitude/longitude (or similar) and is often a better choice for location-based categorical features.
- For more, see: [7.3.4. Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features).
